# TikTok Web BI - ML Category Classifier Training Guide

Notebook ini digunakan untuk melatih model klasifikasi kategori video TikTok berdasarkan teks gabungan (`title_full`, `hashtag_text`, dan `nlp_text`). Model ini menggunakan keluarga algoritma yang sama dengan model-model sebelumnya di project, yaitu kombinasi **TF-IDF Vectorizer** dengan **Linear Support Vector Machine (LinearSVC)** atau **Random Forest Classifier**.

## Langkah-Langkah:
1. Ekspor dataset di local server menggunakan `export_category_dataset.py` yang menghasilkan berkas `category_training_dataset.csv`.
2. Upload berkas `category_training_dataset.csv` ke notebook Google Colab ini.
3. Jalankan seluruh sel di notebook ini untuk melatih model, melakukan evaluasi, dan menyimpan artifact model.
4. Download berkas hasil latihan `category_classifier.pkl` dan letakkan ke folder `ml-service/artifacts/`.
5. Salin output JSON manifest di akhir notebook ini dan tambahkan ke `ml-service/artifacts/registry_manifest.json`.
6. Jalankan script registrasi lokal `python register_colab_artifacts.py` untuk mengaktifkan model baru di database metadata.

### 1. Install Dependencies & Upload Dataset

In [ ]:
# Install library jika belum tersedia
!pip install pandas scikit-learn joblib

In [ ]:
from google.colab import files
import os

if not os.path.exists("category_training_dataset.csv"):
    print("Silakan upload file 'category_training_dataset.csv':")
    uploaded = files.upload()
else:
    print("File category_training_dataset.csv sudah terdeteksi di workspace Colab.")

### 2. Load & Explore Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("category_training_dataset.csv")
print(f"Total data: {df.shape[0]} baris, {df.shape[1]} kolom.\n")
print("Distribusi Kelas Kategori:")
print(df["core_category"].value_counts())
print("\nContoh 3 baris data teratas:")
df.head(3)

### 3. Split Train - Test & Definisikan Pipeline Klasifikasi

Kita membagi data menjadi **80% Training** dan **20% Testing** dengan stratifikasi kelas agar pembagian kategori tetap seimbang. Fitur teks diekstrak menggunakan **TF-IDF Vectorizer** (unigram + bigram, dibatasi 2500 fitur kata teratas).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import joblib

# Pisahkan fitur dan label
X = df["combined_text"].fillna("")
y = df["core_category"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Ukuran data training : {X_train.shape[0]}")
print(f"Ukuran data testing  : {X_test.shape[0]}")

### 4. Pelatihan Model & Evaluasi

Kita akan mencoba dua algoritma klasifikasi (SVM dan Random Forest) yang sudah konsisten dengan arsitektur model sebelumnya di database.

#### A. Model 1: Support Vector Machine (LinearSVC)
LinearSVC sangat optimal dan cepat untuk klasifikasi teks dengan representasi sparse seperti TF-IDF.

In [ ]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2500, ngram_range=(1, 2))),
    ('clf', LinearSVC(C=1.0, class_weight='balanced', random_state=42, dual=False))
])

svm_pipeline.fit(X_train, y_train)
svm_preds = svm_pipeline.predict(X_test)

print("=== LAPORAN EVALUASI SVM ===")
print(classification_report(y_test, svm_preds))

#### B. Model 2: Random Forest Classifier
Sebagai alternatif, kita juga menguji Random Forest Classifier.

In [ ]:
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2500, ngram_range=(1, 2))),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

print("=== LAPORAN EVALUASI RANDOM FOREST ===")
print(classification_report(y_test, rf_preds))

### 5. Pemilihan Model Terbaik & Ekspor Artifact

Umumnya SVM (LinearSVC) mendapatkan performa F1-Score yang jauh lebih tinggi pada klasifikasi teks. Kita akan menyimpan model berkinerja terbaik ke berkas `category_classifier.pkl`.

In [ ]:
# Bandingkan f1-score weighted rata-rata
svm_metrics = precision_recall_fscore_support(y_test, svm_preds, average='weighted')
rf_metrics = precision_recall_fscore_support(y_test, rf_preds, average='weighted')

if svm_metrics[2] >= rf_metrics[2]:
    print("Memilih model: SVM (LinearSVC)")
    best_pipeline = svm_pipeline
    model_type = "svm"
    hyperparams = {
        "C": 1.0,
        "class_weight": "balanced",
        "tfidf_max_features": 2500,
        "tfidf_ngram_range": [1, 2]
    }
    best_preds = svm_preds
else:
    print("Memilih model: Random Forest")
    best_pipeline = rf_pipeline
    model_type = "random_forest"
    hyperparams = {
        "n_estimators": 200,
        "max_depth": 15,
        "class_weight": "balanced",
        "tfidf_max_features": 2500,
        "tfidf_ngram_range": [1, 2]
    }
    best_preds = rf_preds

# Simpan model ke disk
joblib.dump(best_pipeline, "category_classifier.pkl")
print("\n[SUKSES] Berkas 'category_classifier.pkl' berhasil disimpan. Silakan unduh dari folder panel kiri Colab!")

### 6. Pembuatan JSON Metadata Manifest

Jalankan kode di bawah ini, salin JSON hasilnya, lalu tambahkan/sisipkan ke dalam berkas `artifacts/registry_manifest.json` di komputer lokal Anda.

In [ ]:
import json

acc = accuracy_score(y_test, best_preds)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, best_preds, average='weighted')
size_bytes = os.path.getsize("category_classifier.pkl")

manifest_item = {
    "model_name": "category_classifier",
    "model_type": model_type,
    "version": "v1.0",
    "artifact_path": "artifacts/category_classifier.pkl",
    "artifact_size_bytes": size_bytes,
    "metrics": {
        "accuracy_score": round(acc, 4),
        "precision_score": round(precision, 4),
        "recall_score": round(recall, 4),
        "f1_score": round(f1, 4)
    },
    "training_data_size": len(X),
    "feature_columns": [
        "combined_text"
    ],
    "hyperparameters": hyperparams,
    "notes": f"Trained in Google Colab using TF-IDF + {model_type.upper()} on rule_based tagged dataset."
}

print("=== SALIN JSON DI BAWAH INI ===\n")
print(json.dumps(manifest_item, indent=2))
print("\n=== SELESAI ===")